# spaCy: Processamento de Linguagem Natural em Português

Este capítulo transforma o conteúdo do curso em um material de estudo mais completo e tecnicamente atualizado para **JupyterBook**. A ideia não é apenas decorar a API do spaCy, mas entender o **modelo mental** da biblioteca: como texto vira `Doc`, como as anotações linguísticas são produzidas, o que pertence ao `Token`, ao `Span` e ao `Vocab`, e quando usar modelos estatísticos ou regras.

### Objetivos de aprendizagem

Ao final, você deve conseguir:

- explicar a relação entre `Language` (`nlp`), `Doc`, `Token`, `Span`, `Vocab` e `Lexeme`;
- inspecionar tokenização, sentenças, POS, morfologia, lemas e dependências;
- usar NER entendendo que **os rótulos dependem do modelo/idioma**;
- consultar e customizar stopwords sem tratá-las como uma etapa obrigatória;
- entender hashing, vetores estáticos e similaridade por cosseno;
- usar `Matcher`, `PhraseMatcher` e `EntityRuler`;
- visualizar dependências e entidades com `displaCy`;
- otimizar processamento com `nlp.pipe`, `disable` e `select_pipes`;
- posicionar o spaCy em relação a embeddings contextuais, Transformers e LLMs.

**Base:** slides do curso *Formação Processamento de Linguagem Natural, LLMs e GenAI* (tokens; POS/dependências; NER; stopwords; vocabulário; similaridade; Matcher; displaCy; pipelines), complementados pela documentação oficial do spaCy.

## 1. Instalação e carregamento do pipeline em português

O spaCy separa a **biblioteca** (`spacy`) dos **pipelines treinados** (`pt_core_news_sm`, `md`, `lg`). Para este capítulo usaremos `pt_core_news_lg`, porque a versão *large* inclui vetores estáticos de palavras e permite estudar `similarity()` de forma útil.

> **Importante:** nomes como `pt_core_news_lg` descrevem idioma (`pt`), tipo/genre do pipeline (`core_news`) e tamanho (`lg`). Não é seguro inferir detalhes como número de vetores ou rótulos de NER apenas pelo nome; vamos inspecionar o próprio modelo.

In [ ]:
# Execute apenas se necessário no seu ambiente:
# %pip install -U spacy
# !python -m spacy download pt_core_news_lg

In [ ]:
import spacy

print("spaCy:", spacy.__version__)
nlp = spacy.load("pt_core_news_lg")
print("Pipeline:", nlp.pipe_names)
print("Modelo:", nlp.meta.get("name"), nlp.meta.get("version"))
print("Idioma:", nlp.lang)
print("Dimensão dos vetores:", nlp.vocab.vectors_length)
print("Entradas na tabela de vetores:", len(nlp.vocab.vectors))

### `sm`, `md` ou `lg`?

| Variante | Uso típico | Vetores estáticos |
|---|---|---|
| `sm` | menor memória/latência | normalmente não |
| `md` | equilíbrio | sim |
| `lg` | maior cobertura lexical | sim, tabela maior |

O melhor modelo depende da tarefa. **Maior não significa automaticamente melhor** para todo problema.

## 2. Modelo mental: do texto ao `Doc`

A figura abaixo junta duas ideias centrais dos slides: o fluxo `nlp → Doc → Token/Span` e a relação entre `Doc`, `Vocab` e a tabela de strings.

![Arquitetura conceitual do spaCy](attachment:spacy_architecture.svg)

Quando fazemos `doc = nlp(texto)`, acontecem duas fases:

1. o **tokenizer** recebe uma `str` e cria um `Doc`;
2. os componentes do pipeline recebem esse `Doc`, adicionam anotações e o passam adiante.

O tokenizer é especial: ele ocorre **antes** dos componentes listados em `nlp.pipe_names`. Os componentes típicos do modelo em português incluem `tok2vec`, `morphologizer`, `parser`, `lemmatizer`, `attribute_ruler` e `ner`. A ordem importa quando um componente requer anotações criadas por outro, mas não devemos assumir dependências sem verificar.

In [ ]:
print(nlp.pipe_names)

# Mostra o que cada componente atribui/requer.
# É uma forma mais segura de raciocinar sobre dependências do que "chutar".
analysis = nlp.analyze_pipes(pretty=True)

## 3. `Doc`, `Token` e `Span`: três níveis de acesso

- **`Doc`**: documento inteiro e suas anotações.
- **`Token`**: uma posição/token dentro de um `Doc`.
- **`Span`**: uma fatia contínua de um `Doc`; não copia o texto.

A tokenização do spaCy é **não destrutiva**: espaços e offsets são preservados, de modo que `doc.text == texto_original`. Isso é muito importante para alinhar resultados ao texto bruto.

In [ ]:
texto = (
    "As ações do Magazine Luiza S.A., em Franca, Brasil, "
    "caíram 7,5% no trimestre. A empresa anunciou um novo plano."
)

doc = nlp(texto)

print(type(doc))
print("doc.text == texto?", doc.text == texto)
print("Número de tokens:", len(doc))
print("Primeiro token:", doc[0], type(doc[0]))
print("Span:", doc[3:6], type(doc[3:6]))

In [ ]:
# Offsets e espaços permitem reconstruir o texto original.
print(f"{'i':>2} {'token':<18} {'idx':>4} {'whitespace':<12}")
print("-" * 45)
for token in doc[:15]:
    print(f"{token.i:>2} {token.text:<18} {token.idx:>4} {repr(token.whitespace_):<12}")

reconstruido = "".join(t.text_with_ws for t in doc)
print("\nReconstrução perfeita?", reconstruido == texto)

### Segmentação de sentenças

O curso menciona `is_sent_start`; vale completar a ideia com `doc.sents`. No pipeline treinado, os limites de sentença geralmente vêm do parser. Em pipelines leves, podemos usar `sentencizer`.

In [ ]:
for i, sent in enumerate(doc.sents, start=1):
    print(f"Sentença {i}: {sent.text}")

print("\nTokens que iniciam sentença:")
print([t.text for t in doc if t.is_sent_start])

## 4. Atributos lexicais dos tokens

Os slides destacam atributos como stopword, alfanumérico, pontuação e início de sentença. Eles são úteis porque transformam propriedades do texto em **features estruturadas** sem regex manual.

In [ ]:
atributos = []
for token in doc:
    atributos.append({
        "texto": token.text,
        "is_stop": token.is_stop,
        "is_alpha": token.is_alpha,
        "is_punct": token.is_punct,
        "like_num": token.like_num,
        "like_email": token.like_email,
        "shape": token.shape_,
        "sent_start": token.is_sent_start,
    })

for linha in atributos[:18]:
    print(linha)

Alguns atributos úteis para `Matcher` e pré-processamento:

| Atributo | Pergunta respondida |
|---|---|
| `ORTH` / `token.text` | qual é a forma literal? |
| `LOWER` / `token.lower_` | qual é a forma em minúsculas? |
| `IS_ALPHA`, `IS_DIGIT`, `IS_PUNCT` | qual é o tipo ortográfico? |
| `LIKE_NUM`, `LIKE_URL`, `LIKE_EMAIL` | o texto se parece com número/URL/e-mail? |
| `SHAPE` / `shape_` | qual é o padrão visual da palavra? |
| `IS_STOP` / `is_stop` | está na lista de stopwords? |
| `POS`, `DEP`, `LEMMA`, `ENT_TYPE` | qual anotação linguística foi atribuída? |

## 5. POS tagging, morfologia e lematização

O **POS** (*part of speech*) representa a classe gramatical. O spaCy usa os rótulos universais, como `NOUN`, `VERB`, `ADJ`, `ADV`, `PROPN`, `PRON`, `DET`, `ADP`, `AUX`, `CCONJ`, `SCONJ`, `NUM`, `PUNCT` etc.

> **Correção importante:** no `pt_core_news_lg`, o componente principal para POS/morfologia é o `morphologizer`. O pipeline português não precisa ter um `tagger` fino como alguns modelos de inglês. Portanto, `token.pos_` e `token.morph` são as referências mais importantes; `token.tag_` pode não carregar informação útil dependendo do modelo.

In [ ]:
print(f"{'texto':<18} {'POS':<8} {'lema':<18} {'morfologia'}")
print("-" * 90)
for token in doc:
    if not token.is_space:
        print(f"{token.text:<18} {token.pos_:<8} {token.lemma_:<18} {token.morph}")

In [ ]:
# Acesso a traços morfológicos específicos
for token in doc:
    genero = token.morph.get("Gender")
    numero = token.morph.get("Number")
    if genero or numero:
        print(token.text, "Gender=", genero, "Number=", numero)

### Lema não é *stem*

- **Lematização** tenta retornar uma forma lexical válida: `ações → ação`, `caíram → cair`.
- **Stemming** aplica regras de corte e pode produzir radicais que não são palavras.

Para modelos modernos baseados em Transformer, lematizar antes de inferência normalmente **não é necessário** e pode até apagar informação. Para busca, normalização, regras linguísticas e features clássicas, o lema continua sendo muito útil.

## 6. Dependências sintáticas

O parser de dependências representa a sentença como uma árvore dirigida. Para cada token:

- `token.dep_`: relação sintática com o pai;
- `token.head`: token pai;
- `token.children`: filhos diretos;
- `token.subtree`: subárvore inteira;
- o token com `dep_ == "ROOT"` é a raiz da sentença.

Os rótulos do modelo português seguem o esquema de **Universal Dependencies**, com relações como `nsubj`, `obj`, `obl`, `amod`, `det`, `nmod`, `aux` e `ROOT`.

In [ ]:
sent = next(doc.sents)
print("Sentença:", sent.text)
print(f"{'token':<18} {'dep':<14} {'head':<18} {'filhos'}")
print("-" * 80)
for token in sent:
    filhos = ", ".join(c.text for c in token.children)
    print(f"{token.text:<18} {token.dep_:<14} {token.head.text:<18} {filhos}")

roots = [t for t in sent if t.dep_ == "ROOT"]
print("\nROOT:", roots)

### *Noun chunks*

`doc.noun_chunks` retorna sintagmas nominais de base quando o pipeline/idioma oferece essa regra. Eles são `Span`s derivados da análise sintática.

In [ ]:
try:
    for chunk in doc.noun_chunks:
        print(f"{chunk.text:<35} root={chunk.root.text:<15} dep={chunk.root.dep_}")
except NotImplementedError:
    print("Este pipeline/idioma não fornece noun_chunks.")

## 7. Entidades nomeadas (NER)

NER localiza **spans** que representam entidades do mundo real, como pessoas, organizações e lugares. O ponto mais importante é: **o conjunto de rótulos depende do pipeline treinado**.

Os slides do curso mostram uma taxonomia ampla (`PERSON`, `GPE`, `MONEY`, `DATE` etc.), comum em modelos de inglês/OntoNotes. O `pt_core_news_lg` atual para português usa um esquema menor, com rótulos como `PER`, `ORG`, `LOC` e `MISC`. Portanto, nunca copie uma tabela de rótulos de outro idioma sem checar o modelo.

In [ ]:
ner = nlp.get_pipe("ner")
print("Rótulos de NER deste modelo:", ner.labels)

for ent in doc.ents:
    print({
        "texto": ent.text,
        "label": ent.label_,
        "start_token": ent.start,
        "end_token": ent.end,
        "start_char": ent.start_char,
        "end_char": ent.end_char,
    })

> **NER é probabilístico.** Uma entidade pode ser perdida, ter fronteira errada ou receber classe errada. Em produção, valide em dados do seu domínio e combine o modelo com regras (`EntityRuler`) quando existir conhecimento determinístico.

## 8. Stopwords: consultar, adicionar e usar com cuidado

Os slides mostram três operações fundamentais: consultar, adicionar e checar `token.is_stop`. O complemento importante é entender que **remover stopwords não é uma regra universal**.

Em `"não gostei"`, remover `não` pode inverter o sentido. Em Transformers/LLMs, a prática padrão é manter o texto natural. A remoção faz mais sentido em cenários específicos, como algumas representações *bag-of-words* ou regras de busca.

In [ ]:
stops = sorted(nlp.Defaults.stop_words)
print("Total de stopwords:", len(stops))
print("Amostra:", stops[:30])

for palavra in ["de", "não", "dados", "empresa"]:
    lex = nlp.vocab[palavra]
    print(f"{palavra:<10} is_stop={lex.is_stop}")

In [ ]:
# Customização temporária e reversível
palavra_custom = "eita"

nlp.Defaults.stop_words.add(palavra_custom)
nlp.vocab[palavra_custom].is_stop = True
print("Depois de adicionar:", nlp.vocab[palavra_custom].is_stop)

# Limpeza para não deixar o restante do notebook com estado alterado
nlp.Defaults.stop_words.discard(palavra_custom)
nlp.vocab[palavra_custom].is_stop = False
print("Depois de restaurar:", nlp.vocab[palavra_custom].is_stop)

In [ ]:
frase = nlp("Eu não gostei do resultado, mas gostei do atendimento.")
print("Com stopwords: ", [t.text for t in frase if not t.is_punct])
print("Sem stopwords:", [t.text for t in frase if not t.is_stop and not t.is_punct])

## 9. `Vocab`, `StringStore` e `Lexeme`

O slide de vocabulário mostra que o spaCy usa IDs inteiros (hashes) internamente. O `Vocab` é compartilhado entre os `Doc`s produzidos pelo mesmo `nlp` e contém:

- `StringStore`: mapeia string ↔ ID inteiro;
- `Lexeme`: propriedades de uma forma lexical **sem contexto**;
- `Vectors`: tabela de vetores, quando o modelo possui vetores.

A distinção é conceitualmente importante:

- `Token("banco")` conhece contexto, POS, dependência e posição;
- `Lexeme("banco")` representa a forma lexical no vocabulário, sem uma sentença específica.

In [ ]:
hash_id = nlp.vocab.strings["dados"]
print("String → ID:", hash_id)
print("ID → String:", nlp.vocab.strings[hash_id])

lex = nlp.vocab["dados"]
print("\nLexeme:")
print("text:", lex.text)
print("orth:", lex.orth)
print("is_alpha:", lex.is_alpha)
print("is_stop:", lex.is_stop)
print("has_vector:", lex.has_vector)

In [ ]:
doc_a = nlp("dados ajudam decisões")
doc_b = nlp("dados mudam produtos")

print("Mesmo Vocab?", doc_a.vocab is doc_b.vocab is nlp.vocab)
print("Mesmo orth para 'dados'?", doc_a[0].orth == doc_b[0].orth)
print("POS pode depender do contexto?", doc_a[0].pos_, doc_b[0].pos_)

## 10. Vetores e similaridade semântica

O `pt_core_news_lg` inclui **vetores estáticos**. `Token`, `Span` e `Doc` podem expor `.vector` e `.similarity()`; por padrão, a similaridade usa cosseno.

$$
\operatorname{sim}(\mathbf{a},\mathbf{b}) =
\frac{\mathbf{a}\cdot\mathbf{b}}
{\|\mathbf{a}\|\,\|\mathbf{b}\|}
$$

> **Correção conceitual:** o cosseno tem intervalo matemático **[-1, 1]**, não [0, 1]. Em embeddings de linguagem, muitos pares observados ficam positivos, mas valores negativos são possíveis.

In [ ]:
ex = nlp("dados são uma nova forma de ver o mundo")
print("Dimensão de Doc.vector:", ex.vector.shape)
print("Norma do vetor:", ex.vector_norm)

for t in nlp("dados quasarxyz"):
    print(f"{t.text:<12} has_vector={t.has_vector} is_oov={t.is_oov} norm={t.vector_norm:.3f}")

In [ ]:
pares = [
    ("carro", "automóvel"),
    ("carro", "banana"),
    ("ciência de dados", "aprendizado de máquina"),
]

for a, b in pares:
    da, db = nlp(a), nlp(b)
    print(f"{a!r:<28} x {b!r:<28} -> {da.similarity(db):.4f}")

### Estático não é contextual

Os slides associam similaridade a "contexto", mas `pt_core_news_lg` usa vetores estáticos (fastText). Assim, a forma lexical `banco` parte do mesmo vetor-base em "banco financeiro" e "banco da praça". O contexto influencia a média de um `Doc`/`Span`, mas **o vetor lexical da palavra não muda como em BERT**.

Modelos Transformer geram representações contextuais. O spaCy pode integrar Transformers via `spacy-transformers`, mas isso é outro mecanismo e deve ser estudado separadamente.

### Filtrar tokens antes de calcular uma representação: hipótese, não receita

Os slides sugerem remover stopwords, pontuação e pronomes para melhorar similaridade. Isso **pode** ajudar em algumas tarefas, mas precisa ser validado. Uma forma explícita é construir sua própria média de vetores apenas com tokens relevantes.

In [ ]:
import numpy as np

def vetor_filtrado(doc):
    vetores = [t.vector for t in doc if t.has_vector and not t.is_stop and not t.is_punct]
    if not vetores:
        return np.zeros(doc.vocab.vectors_length, dtype="float32")
    return np.mean(vetores, axis=0)

def cosine(a, b):
    den = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / den) if den else 0.0

a = nlp("A empresa apresentou forte crescimento nas vendas")
b = nlp("As vendas da companhia cresceram bastante")

print("spaCy Doc.similarity:", a.similarity(b))
print("Média filtrada:      ", cosine(vetor_filtrado(a), vetor_filtrado(b)))

## 11. `Matcher`: regras sobre tokens

O `Matcher` procura **sequências de tokens descritas por atributos**, como uma expressão regular linguística. Isso o diferencia de `similarity()`:

- `similarity()` retorna um **grau** de proximidade semântica;
- `Matcher` retorna **ocorrências que satisfazem um padrão**.

A grande vantagem sobre regex de caracteres é poder combinar forma (`LOWER`, `ORTH`, `SHAPE`) com propriedades linguísticas (`POS`, `LEMMA`, `DEP`, `ENT_TYPE`) e booleanas (`IS_DIGIT`, `LIKE_EMAIL` etc.).

In [ ]:
from spacy.matcher import Matcher

matcher = Matcher(nlp.vocab)

padrao_telefone = [
    {"ORTH": "("},
    {"SHAPE": "dd"},
    {"ORTH": ")"},
    {"ORTH": "-", "OP": "?"},
    {"IS_DIGIT": True},
]
matcher.add("TELEFONE", [padrao_telefone])

doc_tel = nlp("Ligue para (51) - 9964656570 ou (11) 12344988.")
for match_id, start, end in matcher(doc_tel):
    print("ID numérico:", match_id)
    print("Nome da regra:", nlp.vocab.strings[match_id])
    print("Span:", doc_tel[start:end])

### Operadores (`OP`)

| Operador | Significado |
|---|---|
| `?` | zero ou uma ocorrência |
| `+` | uma ou mais |
| `*` | zero ou mais |
| `!` | deve ocorrer zero vezes naquela posição |

Também podemos usar comparadores como `IN`, `NOT_IN`, `>=`, `<=` e padrões regex dentro de atributos.

In [ ]:
matcher_tec = Matcher(nlp.vocab)
matcher_tec.add("TEC", [[
    {"LOWER": {"IN": ["machine", "deep"]}},
    {"LOWER": "learning"},
]])

doc_tec = nlp("Machine learning e deep learning são subáreas importantes.")
spans = matcher_tec(doc_tec, as_spans=True)
print(spans)

## 12. `PhraseMatcher` e `EntityRuler`: quando regras simples bastam

O curso foca no `Matcher`, mas duas ferramentas completam o quadro:

- **`PhraseMatcher`**: melhor para listas de frases/termos exatos, como nomes de produtos, cidades ou competências;
- **`EntityRuler`**: transforma regras em entidades de `Doc.ents` e pode ser combinado com o NER estatístico.

Use regra quando o conhecimento for determinístico e estável; use modelo quando houver variação linguística que seria difícil enumerar.

In [ ]:
from spacy.matcher import PhraseMatcher

phrase_matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
termos = ["machine learning", "ciência de dados", "inteligência artificial"]
patterns = [nlp.make_doc(t) for t in termos]
phrase_matcher.add("AREA", patterns)

doc_frases = nlp("Ciência de dados usa machine learning e estatística.")
for span in phrase_matcher(doc_frases, as_spans=True):
    print(span.text, span.label_)

In [ ]:
# EntityRuler: regra que entra no mesmo espaço de entidades do NER.
# Usamos after="ner" + overwrite_ents=True para regras de domínio terem prioridade.
if "entity_ruler" in nlp.pipe_names:
    nlp.remove_pipe("entity_ruler")

ruler = nlp.add_pipe("entity_ruler", after="ner", config={"overwrite_ents": True})
ruler.add_patterns([
    {"label": "TECNOLOGIA", "pattern": "spaCy"},
    {"label": "TECNOLOGIA", "pattern": "PyTorch"},
    {"label": "TECNOLOGIA", "pattern": [{"LOWER": "machine"}, {"LOWER": "learning"}]},
])

doc_ruler = nlp("Usamos spaCy com PyTorch em um projeto de machine learning.")
print([(ent.text, ent.label_) for ent in doc_ruler.ents])

# Restaurar o pipeline base para as seções seguintes.
nlp.remove_pipe("entity_ruler")

> Para padrões definidos pela **estrutura da árvore sintática**, existe ainda o `DependencyMatcher`. Ele é útil quando a relação entre palavras importa mais do que a adjacência.

## 13. Visualização com `displaCy`

O `displaCy` fornece os dois estilos destacados nos slides:

- `style="ent"`: entidades nomeadas;
- `style="dep"`: árvore de dependências.

Essas visualizações são especialmente úteis para **debugging de modelos**, porque tornam erros de fronteira, rótulo e análise sintática visíveis.

In [ ]:
from spacy import displacy

displacy.render(doc, style="ent", jupyter=True)

In [ ]:
displacy.render(
    next(doc.sents),
    style="dep",
    jupyter=True,
    options={"compact": True, "distance": 90}
)

## 14. Pipeline: habilitar só o necessário

Um pipeline treinado pode executar vários componentes caros. Em vez de remover e recriar componentes no meio do notebook, prefira formas **seguras** de desativação:

1. carregar já com `disable=[...]`;
2. usar `nlp.select_pipes(disable=[...])` temporariamente;
3. processar coleções com `nlp.pipe()`.

> Evite remover `tok2vec` e depois adicionar um novo `tok2vec` achando que restaurou o original: o novo componente não carrega automaticamente os mesmos pesos/conexões do pipeline treinado.

In [ ]:
print("Pipeline completo:", nlp.pipe_names)

# Desativação temporária e reversível
with nlp.select_pipes(disable=["parser", "ner"]):
    doc_leve = nlp("Este texto não precisa de dependências nem entidades.")
    print("Dentro do contexto:", nlp.pipe_names)

print("Depois do contexto: ", nlp.pipe_names)

In [ ]:
# Outra opção: carregar uma instância separada já sem componentes desnecessários.
nlp_sem_ner_parser = spacy.load("pt_core_news_lg", disable=["parser", "ner"])
print(nlp_sem_ner_parser.pipe_names)

### `nlp.pipe`: processamento em lote

Os slides citam o *pipe mode*: `nlp.pipe()` é um gerador e evita a sobrecarga de chamar `nlp(texto)` repetidamente. `batch_size` controla os lotes e `n_process` pode habilitar multiprocessamento.

Multiprocessamento não é automaticamente mais rápido: modelos grandes, overhead de processos e memória podem inverter o ganho. Meça no seu ambiente.

In [ ]:
textos = [
    "A empresa divulgou seus resultados.",
    "O mercado reagiu positivamente.",
    "Os analistas revisaram suas projeções.",
] * 100

for i, d in enumerate(nlp.pipe(textos, batch_size=64, n_process=1)):
    if i < 3:
        print(d.text, "->", [(e.text, e.label_) for e in d.ents])

## 15. Pipeline customizado

O spaCy também permite criar componentes próprios. Um componente recebe um `Doc`, adiciona informação e devolve o mesmo `Doc`. Para guardar dados customizados, usamos extensões com `Doc.set_extension`, `Token.set_extension` ou `Span.set_extension`.

In [ ]:
from spacy.language import Language
from spacy.tokens import Doc

if not Doc.has_extension("n_alpha"):
    Doc.set_extension("n_alpha", default=0)

if not Language.has_factory("conta_tokens_alpha"):
    @Language.component("conta_tokens_alpha")
    def conta_tokens_alpha(doc):
        doc._.n_alpha = sum(t.is_alpha for t in doc)
        return doc

nlp_custom = spacy.blank("pt")
nlp_custom.add_pipe("conta_tokens_alpha")

doc_custom = nlp_custom("spaCy 3 processa texto muito rápido!")
print("Tokens alfabéticos:", doc_custom._.n_alpha)
print("Pipeline customizado:", nlp_custom.pipe_names)

## 16. Miniaplicação: combinar modelo estatístico + regras

Um padrão muito útil em projetos reais é usar:

- **NER estatístico** para entidades com variação de escrita;
- **Matcher** para estruturas determinísticas do domínio.

No pipeline português, por exemplo, o NER não possui necessariamente rótulos `MONEY` e `PERCENT`. Podemos extraí-los por regra sem treinar um novo NER.

In [ ]:
texto_negocio = (
    "A Vale anunciou investimento de R$ 12 bilhões em Minas Gerais, "
    "com crescimento projetado de 8% para o próximo ciclo."
)
doc_negocio = nlp(texto_negocio)

matcher_kpi = Matcher(nlp.vocab)
matcher_kpi.add("PERCENTUAL", [[{"LIKE_NUM": True}, {"ORTH": "%"}]])
matcher_kpi.add("DINHEIRO", [
    [{"TEXT": {"REGEX": r"(?i)^R\$$"}}, {"LIKE_NUM": True},
     {"LOWER": {"IN": ["mil", "milhão", "milhões", "bilhão", "bilhões"]}, "OP": "?"}],
    [{"LOWER": "r"}, {"ORTH": "$"}, {"LIKE_NUM": True},
     {"LOWER": {"IN": ["mil", "milhão", "milhões", "bilhão", "bilhões"]}, "OP": "?"}],
])

print("NER do modelo:", [(e.text, e.label_) for e in doc_negocio.ents])
print("Regras:", [(s.text, s.label_) for s in matcher_kpi(doc_negocio, as_spans=True)])

### Por que essa combinação é forte?

NER resolve casos flexíveis e linguísticos; regras cobrem formatos que você conhece exatamente. Em produção, isso costuma ser mais auditável do que tentar forçar tudo para dentro de um único modelo.

## 17. Onde o spaCy entra no caminho até Transformers e LLMs?

O spaCy é uma biblioteca de **NLP/PLN industrial**. Ele não é um LLM. Seu valor está em:

- tokenização e alinhamento de texto;
- anotações linguísticas estruturadas;
- NER e parsing;
- regras linguísticas eficientes;
- pipelines reprodutíveis e rápidos em CPU;
- integração com modelos Transformer quando necessário.

Uma progressão conceitual útil é:

**texto → tokenização → representações → análise linguística → embeddings estáticos → embeddings contextuais → atenção → Transformers → LLMs**.

Este capítulo cobre principalmente a parte de **estruturação e análise do texto**, além de introduzir embeddings estáticos. O salto para BERT/Transformers muda a forma como as representações são construídas: deixam de ser vetores fixos por palavra e passam a depender fortemente do contexto.

## 18. Erros e armadilhas comuns

1. **Confundir rótulos entre idiomas.** `PERSON/GPE/MONEY` de um modelo inglês não implica os mesmos rótulos em português.
2. **Tratar `tag_` como universal.** O que existe depende do componente treinado; em português, foque `pos_` + `morph`.
3. **Dizer que cosseno vai de 0 a 1.** O intervalo matemático é `[-1, 1]`.
4. **Chamar vetores estáticos de contextuais.** O contexto do `Doc` pode alterar a média, mas o vetor lexical de uma palavra continua estático.
5. **Remover stopwords automaticamente.** Valide por tarefa; negação e relações sintáticas podem ser essenciais.
6. **Modificar um pipeline treinado de modo destrutivo.** Para performance, prefira `disable`, `select_pipes` e `nlp.pipe`.
7. **Assumir que NER é verdade factual.** É uma predição estatística e deve ser avaliada no domínio.
8. **Usar `Matcher` quando uma lista de frases resolveria melhor.** Para dicionários grandes, `PhraseMatcher` costuma ser a escolha natural.

## 19. Exercícios de fixação

### Exercício 1 — Tokenização e offsets
Escolha uma frase com números, siglas e pontuação. Mostre `text`, `idx`, `whitespace_` e prove que a reconstrução com `text_with_ws` é idêntica ao texto original.

### Exercício 2 — Morfologia
Encontre todos os verbos de um parágrafo e construa uma tabela com `text`, `lemma_`, `VerbForm`, `Tense`, `Mood` e `Person` quando disponíveis.

### Exercício 3 — Dependências
Para uma frase simples, encontre o `ROOT`, seus filhos e a subárvore do sujeito (`nsubj`). Renderize com `displaCy`.

### Exercício 4 — NER
Rode cinco frases brasileiras sobre empresas, pessoas e cidades. Compare os rótulos reais do modelo com o que você esperava e registre falsos positivos/negativos.

### Exercício 5 — Matcher
Crie regras para detectar pelo menos dois formatos de CPF, CNPJ, telefone ou percentual. Use `as_spans=True`.

### Exercício 6 — Regras + NER
Adicione um `EntityRuler` para 10 termos específicos de um domínio e compare o resultado antes/depois.

### Exercício 7 — Performance
Compare o tempo de `for texto in textos: nlp(texto)` com `nlp.pipe(textos, batch_size=...)`. Depois desative `parser` e `ner` e meça de novo.

## 20. Resumo

| Conceito | API principal | Ideia-chave |
|---|---|---|
| Pipeline | `nlp`, `pipe_names`, `analyze_pipes()` | texto → `Doc` anotado |
| Tokenização | `Doc`, `Token`, `idx`, `whitespace_` | não destrutiva |
| Span | `doc[a:b]`, `Span` | visão contínua do `Doc` |
| Sentenças | `doc.sents`, `is_sent_start` | limites dependem do pipeline |
| POS | `token.pos_` | classe gramatical universal |
| Morfologia | `token.morph` | gênero, número, tempo etc. |
| Lema | `token.lemma_` | forma canônica |
| Dependências | `dep_`, `head`, `children`, `subtree` | árvore sintática |
| NER | `doc.ents`, `ent.label_` | rótulos dependem do modelo |
| Stopwords | `is_stop`, `Defaults.stop_words` | uso dependente da tarefa |
| Vocabulário | `nlp.vocab`, `StringStore`, `Lexeme` | dados lexicais compartilhados |
| Vetores | `.vector`, `.has_vector`, `.is_oov` | representação estática no `lg` |
| Similaridade | `.similarity()` | cosseno, `[-1,1]` |
| Matcher | `Matcher` | padrões token a token |
| Frases | `PhraseMatcher` | listas de expressões |
| Entidades por regra | `EntityRuler` | regras integradas ao `Doc.ents` |
| Visualização | `displacy.render()` | entidades e dependências |
| Lote/performance | `nlp.pipe()` | batching e multiprocessamento |
| Pipeline leve | `disable`, `select_pipes()` | execute só o necessário |

## 21. Referências para aprofundamento

- spaCy — **Linguistic Features**: https://spacy.io/usage/linguistic-features
- spaCy — **Language Processing Pipelines**: https://spacy.io/usage/processing-pipelines
- spaCy — **Rule-based matching**: https://spacy.io/usage/rule-based-matching
- spaCy — **Portuguese trained pipelines**: https://spacy.io/models/pt
- spaCy API — **Vocab / Lexeme / Doc / Token / Span**: https://spacy.io/api

> As APIs e rótulos podem variar entre versões e pipelines. Para notebooks de estudo duráveis, prefira inspecionar `nlp.meta`, `nlp.pipe_names`, `nlp.analyze_pipes()` e os `.labels` dos componentes em vez de depender de listas copiadas de outro modelo.